## Libraries and data

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

c:\Users\Joaquin\Documents\GitHub\skforecast
0.23.0


In [2]:
# Libraries
# ==============================================================================
import os
import pandas as pd
import torch
import time
import matplotlib.pyplot as plt
from skforecast.datasets import fetch_dataset
from skforecast.foundation import FoundationModel, ForecasterFoundation
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_foundation,
    bayesian_search_foundation
)
from skforecast.plot import set_dark_theme

color = '\033[1m\033[38;5;208m' 
print(f"{color}torch version: {torch.__version__}")
print(f"  Cuda available : {torch.cuda.is_available()}")
print(f"  MPS available  : {torch.backends.mps.is_available()}")

torch version: 2.12.1+cu126
  Cuda available : True
  MPS available  : False


In [3]:
# Data download
# ==============================================================================
data = fetch_dataset(name='vic_electricity')

# Aggregating in 1H intervals
# ==============================================================================
# The Date column is eliminated so that it does not generate an error when aggregating.
data = data.drop(columns="Date")
data = (
    data
    .resample(rule="h", closed="left", label="right")
    .agg({
        "Demand": "mean",
        "Temperature": "mean",
        "Holiday": "mean",
    })
)
data.head(3)

╭──────────────────────────── vic_electricity ─────────────────────────────╮
│ Description:                                                             │
│ Half-hourly electricity demand for Victoria, Australia                   │
│                                                                          │
│ Source:                                                                  │
│ O'Hara-Wild M, Hyndman R, Wang E, Godahewa R (2022).tsibbledata: Diverse │
│ Datasets for 'tsibble'. https://tsibbledata.tidyverts.org/,              │
│ https://github.com/tidyverts/tsibbledata/.                               │
│ https://tsibbledata.tidyverts.org/reference/vic_elec.html                │
│                                                                          │
│ URL:                                                                     │
│ https://raw.githubusercontent.com/skforecast/skforecast-                 │
│ datasets/main/data/vic_electricity.csv                                   │
│                                                                          │
│ Shape: 52608 rows x 4 columns                                            │
╰──────────────────────────────────────────────────────────────────────────╯

,Demand,Temperature,Holiday
Time,,,
2011-12-31 14:00:00,4323.095350,21.225,1.0
2011-12-31 15:00:00,3963.264688,20.625,1.0
2011-12-31 16:00:00,3950.913495,20.325,1.0


In [4]:
# Split data into train-test
# ==============================================================================
data = data.loc['2012-01-01 00:00:00':'2014-12-30 23:00:00', :].copy()
end_train = '2014-11-30 23:59:00'
data_train = data.loc[: end_train, :].copy()
data_test  = data.loc[end_train:, :].copy()

print(f"Train dates: {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Test dates : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")


Train dates: 2012-01-01 00:00:00 --- 2014-11-30 23:00:00  (n=25560)
Test dates : 2014-12-01 00:00:00 --- 2014-12-30 23:00:00  (n=720)


In [6]:
cv = TimeSeriesFold(
         steps              = 24,
         initial_train_size = len(data.loc[:end_train]),
         refit              = False
     )

estimator = FoundationModel(model_id="autogluon/chronos-2-small", context_length=150)
forecaster = ForecasterFoundation(estimator=estimator)


# Search space
def search_space(trial):
    search_space  = {
        'context_length' : trial.suggest_int('context_length', 100, 500),
    }
    
    return search_space

results, study = bayesian_search_foundation(
    forecaster=forecaster,
    series=data['Demand'],
    cv = cv,
    search_space = search_space,
    exog= data[['Temperature', 'Holiday']],
    metric = 'mean_absolute_error',
    n_trials= 5,

)

  0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/92 [00:00<?, ?it/s]

In [7]:
results

,trial_number,levels,params,mean_absolute_error,context_length
0,4,[Demand],{'context_length': 388},168.273552,388
1,0,[Demand],{'context_length': 379},169.276201,379
2,3,[Demand],{'context_length': 321},174.177127,321
3,1,[Demand],{'context_length': 214},178.322617,214
4,2,[Demand],{'context_length': 190},195.160782,190


In [5]:
# Data
# ==============================================================================
data_multiseries = fetch_dataset(name="items_sales")
display(data_multiseries.head(3))

╭─────────────────────── items_sales ───────────────────────╮
│ Description:                                              │
│ Simulated time series for the sales of 3 different items. │
│                                                           │
│ Source:                                                   │
│ Simulated data.                                           │
│                                                           │
│ URL:                                                      │
│ https://raw.githubusercontent.com/skforecast/skforecast-  │
│ datasets/main/data/simulated_items_sales.csv              │
│                                                           │
│ Shape: 1097 rows x 3 columns                              │
╰───────────────────────────────────────────────────────────╯

,item_1,item_2,item_3
date,,,
2012-01-01,8.253175,21.047727,19.429739
2012-01-02,22.777826,26.578125,28.009863
2012-01-03,27.549099,31.751042,32.078922


In [6]:
# Split data into train-test
# ==============================================================================
end_train = '2014-07-15 23:59:00'
data_multiseries_train = data_multiseries.loc[:end_train, :].copy()
data_multiseries_test  = data_multiseries.loc[end_train:, :].copy()

In [13]:
cv = TimeSeriesFold(
         steps              = 24,
         initial_train_size = len(data_multiseries.loc[:end_train]),
         refit              = False
     )

estimator = FoundationModel(model_id="autogluon/chronos-2-small", context_length=150)
forecaster = ForecasterFoundation(estimator=estimator)


# Search space
def search_space(trial):
    search_space  = {
        'context_length' : trial.suggest_int('context_length', 100, 1000, step=100),
    }
    
    return search_space

results, study = bayesian_search_foundation(
    forecaster=forecaster,
    series=data_multiseries,
    cv = cv,
    search_space = search_space,
    metric = 'mean_absolute_error',
    n_trials= 20,
    suppress_warnings = True
)

  0%|          | 0/20 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/92 [00:00<?, ?it/s]

In [14]:
results

,trial_number,levels,params,mean_absolute_error__weighted_average,mean_absolute_error__average,mean_absolute_error__pooling,context_length
0,4,"[item_1, item_2, item_3]",{'context_length': 800},2.016128,2.016128,2.016128,800
1,14,"[item_1, item_2, item_3]",{'context_length': 800},2.016128,2.016128,2.016128,800
2,12,"[item_1, item_2, item_3]",{'context_length': 800},2.016128,2.016128,2.016128,800
3,17,"[item_1, item_2, item_3]",{'context_length': 800},2.016128,2.016128,2.016128,800
4,8,"[item_1, item_2, item_3]",{'context_length': 500},2.018613,2.018613,2.018613,500
5,5,"[item_1, item_2, item_3]",{'context_length': 500},2.018613,2.018613,2.018613,500
6,13,"[item_1, item_2, item_3]",{'context_length': 900},2.021500,2.021500,2.021500,900
7,10,"[item_1, item_2, item_3]",{'context_length': 900},2.021500,2.021500,2.021500,900
8,15,"[item_1, item_2, item_3]",{'context_length': 1000},2.021701,2.021701,2.021701,1000
9,6,"[item_1, item_2, item_3]",{'context_length': 1000},2.021701,2.021701,2.021701,1000
